In [0]:
%sql
USE CATALOG futebol_analytics;
USE SCHEMA gold;

In [0]:
%sql
SELECT 
    COALESCE(c_origem.domestic_competition_id, 'Divisões Menores / Outras') AS liga_origem,
    COUNT(*) AS total_contratacoes,
    ROUND(AVG(f.transfer_fee) / 1000000, 2) AS taxa_media_milhoes_eur,
    ROUND(AVG(f.minutos_jogados_ano_t), 0) AS media_minutos_ano_chegada,
    ROUND(AVG(f.nota_media_ano_t), 2) AS nota_media_ano_chegada,
    ROUND(100.0 * SUM(f.flag_sucesso_consolidado) / COUNT(*), 1) AS taxa_sucesso_pct
FROM fato_transferencias_desempenho f
LEFT JOIN dim_clubes c_origem ON f.from_club_id = c_origem.club_id
WHERE f.nota_media_ano_t IS NOT NULL
GROUP BY 1
HAVING COUNT(*) >= 5
ORDER BY taxa_sucesso_pct DESC;

In [0]:
%sql
SELECT 
    CASE 
        WHEN (f.ano_transferencia - YEAR(d.date_of_birth)) <= 21 THEN '1. Sub-21 (Potencial / Formação)'
        WHEN (f.ano_transferencia - YEAR(d.date_of_birth)) BETWEEN 22 AND 25 THEN '2. 22-25 anos (Desenvolvimento)'
        WHEN (f.ano_transferencia - YEAR(d.date_of_birth)) BETWEEN 26 AND 29 THEN '3. 26-29 anos (Pico de Maturação)'
        ELSE '4. 30+ anos (Veteranos / Experiência)'
    END AS faixa_etaria_contratacao,
    COUNT(*) AS total_atletas,
    ROUND(AVG(f.transfer_fee) / 1000000, 2) AS custo_medio_milhoes_eur,
    ROUND(AVG(f.minutos_jogados_ano_t), 0) AS media_minutos_chegada,
    ROUND(AVG(f.nota_media_ano_t), 2) AS nota_media_tecnica,
    ROUND(100.0 * SUM(f.flag_sucesso_consolidado) / COUNT(*), 1) AS taxa_sucesso_pct
FROM fato_transferencias_desempenho f
INNER JOIN dim_jogadores d ON f.player_id = d.player_id
WHERE d.date_of_birth IS NOT NULL 
  AND f.nota_media_ano_t IS NOT NULL
GROUP BY 1
ORDER BY 1 ASC;

In [0]:
%sql
SELECT 
    CASE 
        WHEN f.minutos_jogados_t_minus_1 >= 2000 THEN '1. Titular Absoluto no Clube de Origem (>= 2000 min)'
        WHEN f.minutos_jogados_t_minus_1 BETWEEN 1000 AND 1999 THEN '2. Rotatividade Regular (1000 - 1999 min)'
        WHEN f.minutos_jogados_t_minus_1 > 0 THEN '3. Pouca Rodagem (< 1000 min)'
        ELSE '4. Sem Minutos Registrados / Inativo'
    END AS perfil_temporada_anterior,
    COUNT(*) AS total_contratacoes,
    ROUND(AVG(f.minutos_jogados_ano_t), 0) AS media_minutos_ano_t,
    ROUND(AVG(f.nota_media_ano_t), 2) AS nota_media_ano_t,
    ROUND(100.0 * SUM(f.flag_sucesso_fisico) / COUNT(*), 1) AS taxa_aproveitamento_fisico_pct,
    ROUND(100.0 * SUM(f.flag_sucesso_consolidado) / COUNT(*), 1) AS taxa_sucesso_consolidado_pct
FROM fato_transferencias_desempenho f
WHERE f.nota_media_ano_t IS NOT NULL
GROUP BY 1
ORDER BY 1 ASC;

In [0]:
%sql
SELECT 
    CASE 
        WHEN f.flag_sucesso_consolidado = 1 THEN 'Contratação Bem-Sucedida' 
        ELSE 'Contratação Abaixo do Esperado' 
    END AS status_contratacao,
    COUNT(*) AS total_avaliado,
    ROUND(AVG(f.transfer_fee) / 1000000, 2) AS valor_medio_investido_milhoes,
    ROUND(AVG(f.ano_transferencia - YEAR(d.date_of_birth)), 1) AS idade_media,
    ROUND(AVG(f.minutos_jogados_t_minus_1), 0) AS media_minutos_ano_anterior,
    ROUND(AVG(f.nota_media_ano_t), 2) AS nota_media_ano_t,
    ROUND(AVG(f.gols_assistencias_ano_t), 1) AS media_participacoes_gols
FROM fato_transferencias_desempenho f
INNER JOIN dim_jogadores d ON f.player_id = d.player_id
WHERE f.transfer_fee BETWEEN 5000000 AND 25000000
  AND f.nota_media_ano_t IS NOT NULL
GROUP BY 1;